| TESTE BERT BASE|
Cálculo SPAN com 3 épocas e 30 experimentos no Collab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
caminho = '/content/drive/MyDrive/Colab Notebooks'
os.chdir(caminho)

In [ ]:
# Update e instalação dos pacotes e dependências usados

!pip install -q transformers datasets evaluate seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


In [ ]:
# Importações necessárias

import pandas as pd
import csv
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from datasets import concatenate_datasets

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import evaluate
metric = evaluate.load("seqeval")

In [ ]:
# Definição dos rótulos BIO
lista_rotulos = ["O", "B-WRONG", "I-WRONG"]
label_to_id = {label: i for i, label in enumerate(lista_rotulos)}
id_to_label = {i: label for i, label in enumerate(lista_rotulos)}

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove os índices de ignorados (-100) e converte para labels de texto
    true_predictions = [
        [id_to_label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id_to_label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    precision = results["overall_precision"]
    recall = results["overall_recall"]

    # F0.5 Score
    if precision + recall == 0: # Para evitar divisões por 0
        f0_5=0.0

    else:
        f0_5= (1+0.5**2) * (precision*recall) / ((0.5**2 * precision) + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f0_5": f0_5,
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
import torch
device = torch.device('cuda')
print(f"Usando dispositivo: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

Usando dispositivo: cuda
PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8


In [ ]:
# Lê arquivo CONLL (TSV)

def ler_conll(
    path: str,
    *,
    ignore_header_regex: str = r"^\s*Sentença\s+Original\s+N[°º]\s*\d+",
    allow_space_in_token: bool = False,
    strict_two_columns: bool = False,
    verbose: bool = True,
) -> Tuple[List[List[str]], List[List[str]]]:
    header_pat = re.compile(ignore_header_regex, flags=re.IGNORECASE)
    tokens_per_sent, tags_per_sent = [], []
    cur_tokens, cur_tags = [], []

    def flush_sentence():
        nonlocal cur_tokens, cur_tags
        if cur_tokens:
            tokens_per_sent.append(cur_tokens)
            tags_per_sent.append(cur_tags)
        cur_tokens, cur_tags = [], []

    with open(path, "r", encoding="utf-8") as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.rstrip("\n")

            # Ignora linhas de cabeçalho por sentença
            if header_pat.search(line):
                flush_sentence()
                continue

            # Delimitador de sentença
            if line.strip() == "":
                flush_sentence()
                continue

            # Tenta split por TAB primeiro, senão por espaço
            if "\t" in line:
                parts = line.split("\t")
            else:
                # Divide por qualquer espaço
                parts = line.strip().split()

            if strict_two_columns:
                if len(parts) != 2:
                    if verbose:
                        print(f"[AVISO] Linha {lineno} ignorada (esperado 2 colunas): {line}")
                    continue
                token, tag = parts[0], parts[1]
            else:
                if len(parts) < 2:
                    if verbose:
                        print(f"[AVISO] Linha {lineno} ignorada (colunas insuficientes): {line}")
                    continue
                if allow_space_in_token:
                    # Última coluna é tag; o resto é token (pode conter espaços/tabs)
                    *token_parts, tag = parts
                    token = " ".join(token_parts)
                else:
                    # Primeira é token, última é tag (se houver colunas no meio, descartamos/avisamos)
                    token, tag = parts[0], parts[-1]
                    if len(parts) > 2 and verbose:
                        print(f"[OBSERVAÇÃO] Linha {lineno} possuía colunas extras; usando 1ª como token e última como tag.")

            # Sanidade básica da tag
            if not tag or tag.isspace():
                if verbose:
                    print(f"[AVISO] Linha {lineno} com tag vazia; ignorada: {line}")
                continue

            cur_tokens.append(token)
            cur_tags.append(tag)

    # Última sentença (se não terminar com linha em branco)
    flush_sentence()

    return tokens_per_sent, tags_per_sent

In [ ]:
# Mapeia Label2ID e ID2Label

def build_label_maps(tags_per_sent: List[List[str]]) -> Tuple[Dict[str, int], Dict[int, str]]:
    labels = set()
    for sent_tags in tags_per_sent:
        labels.update(sent_tags)

    labels = list(labels)

    if "O" in labels:
        labels.remove("O")
        labels = ["O"] + sorted(labels)
    else:
        labels = sorted(labels)

    label2id = {lab: i for i, lab in enumerate(labels)} # Converte texto em número ['O': 0, 'B-WRONG': 1, 'I-WRONG': 2]
    id2label = {i: lab for lab, i in label2id.items()} # Converte número em texto [0: 'O', 1: 'B-WRONG', 2: 'I-WRONG']
    return label2id, id2label

In [ ]:
# Constrói um dataset Hugging Face com colunas de tokens (List[str]) e ner_tags (List[int]).

def make_hf_dataset_from_tokens(
    tokens_per_sent: List[List[str]],
    tags_per_sent: List[List[str]],
    label2id: Optional[Dict[str, int]] = None,
) -> Tuple[Dataset, Dict[str, int]]:

    # Verifica se as sentenças de tokens e as sentenças de tags são equivalentes (mesmo número)
    assert len(tokens_per_sent) == len(tags_per_sent), "tokens e tags desalinhados"

    # Associa Label2ID ao mapeamento das tags feito anteriormente
    if label2id is None:
        label2id, _ = build_label_maps(tags_per_sent)

    # Tags desconhecidas serão associadas à tag O
    O_ID = label2id.get("O", 0)

    # Mapeamento com tratamento de erro
    encoded_tags = []
    for tags in tags_per_sent:
        sentence_tags = []
        for tag in tags:
            sentence_tags.append(label2id.get(tag, O_ID)) # Verifica se tag está mapeada ou se uso o ID 0
        encoded_tags.append(sentence_tags)

    ds = Dataset.from_dict({"tokens": tokens_per_sent, "ner_tags": encoded_tags})
    return ds, label2id

In [ ]:
# Função para equiparar lista de labels com lista de tokens

def tokenize_and_align_labels(batch, tokenizer, label_all_subtokens: bool = False):
    tokenized = tokenizer(
        batch["tokens"],
        truncation=True,
        padding=True,
        is_split_into_words=True,
    )

    all_labels = []
    for i, word_labels in enumerate(batch["ner_tags"]):
        # Mapeia o subtoken com a palavra original
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                # Caractere especial é ignorado
                label_ids.append(-100)
            elif word_id != previous_word_id:
                # Primeiro pedaço da palavra recebe tag original
                label_ids.append(word_labels[word_id])
            else:
                # Sub-token da mesma palavra (word) é alinhado à label original
                label_ids.append(word_labels[word_id] if label_all_subtokens else -100)
            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized


In [ ]:
# Função que retorna o objeto DatasetDict para o treinamento do modelo a partir das funções anteriores.

def load_custom_conll_dataset(
    files: Dict[str, str] | str,
    *,
    fixed_label2id: Optional[Dict[str, int]] = None,
    allow_space_in_token: bool = False,
    strict_two_columns: bool = False,
    verbose: bool = True,
) -> Tuple[DatasetDict, Dict[str, int], Dict[int, str]]:

    if isinstance(files, str):
        files = {"train": files}

    datasets_splits = {}
    cache_tokens = {}
    cache_tags = {}

    # Lê os splits
    for split_name, path in files.items():
        tokens_per_sent, tags_per_sent = ler_conll(
            path,
            allow_space_in_token=allow_space_in_token,
            strict_two_columns=strict_two_columns,
            verbose=verbose,
        )
        cache_tokens[split_name] = tokens_per_sent
        cache_tags[split_name] = tags_per_sent

    # Determinar o mapeamento de labels
    if fixed_label2id is not None:
        # Mapeamento fixo (BIO)
        accumulated_label2id = fixed_label2id
    else:
        # Remapeaia todas as tags caso não tenha labels definidas (fallback)
        all_tags = []
        for split_name in files.keys():
            all_tags.extend(cache_tags[split_name])
        accumulated_label2id, _ = build_label_maps(all_tags)

    accumulated_id2label = {i: lab for lab, i in accumulated_label2id.items()}

    # Constrói datasets usando o mapeamento (BIO)
    for split_name in files.keys():
        ds, _ = make_hf_dataset_from_tokens(
            cache_tokens[split_name],
            cache_tags[split_name],
            label2id=accumulated_label2id,
        )
        datasets_splits[split_name] = ds

    dset = DatasetDict(datasets_splits)
    return dset, accumulated_label2id, accumulated_id2label

In [ ]:
# Carregamento do Dataset já com os labels BIO
path_treino = "train_bio.tsv"
path_teste = "test_bio.tsv"
path_validacao = "val_bio.tsv"

dataset, _, _ = load_custom_conll_dataset(
      files={"train": path_treino, "test": path_teste, "validation": path_validacao},
      fixed_label2id=label_to_id,
      allow_space_in_token=False,
      strict_two_columns=False,
      verbose=True,
)

# Unificação dos dados para que o sorteio 80/20 seja global a cada rodada
full_dataset = concatenate_datasets([
    dataset["train"],
    dataset["test"],
    dataset["validation"]
])

num_experimentos = 30
resultados_experimentos = [] # Lista para armazenar métricas de cada rodada

print(full_dataset)
print("labels:", label_to_id)

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 19250
})
labels: {'O': 0, 'B-WRONG': 1, 'I-WRONG': 2}


In [ ]:
# 1. Verify id2label keys are integers
print("id2label key types:", [type(k) for k in id_to_label.keys()])

# 2. Verify label_to_id values are integers
print("label_to_id value types:", [type(v) for v in label_to_id.values()])

# 3. Check for consistency
print(f"Number of labels in list: {len(lista_rotulos)}")
print(f"Number of entries in id2label: {len(id_to_label)}")

id2label key types: [<class 'int'>, <class 'int'>, <class 'int'>]
label_to_id value types: [<class 'int'>, <class 'int'>, <class 'int'>]
Number of labels in list: 3
Number of entries in id2label: 3


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

# Tokenização + alinhamento das labels
tokenized_dataset = dataset.map(
    lambda batch: tokenize_and_align_labels(batch, tokenizer, label_all_subtokens=False), # Aplica a função para alinhamento de tokens e tags
    batched=True # Processa várias sentenças simultaneamente
)

# Carregamento do Modelo
model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=len(lista_rotulos),
    id2label=id_to_label,
    label2id=label_to_id,
    use_safetensors=True
)

model.to(device)

print(f"\nModelo movido para: {device}")
print(f"\nOs rótulos foram definidos como: \n{model.config.id2label}")

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Map:   0%|          | 0/3850 [00:00<?, ? examples/s]

Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Modelo movido para: cuda

Os rótulos foram definidos como: 
{0: 'O', 1: 'B-WRONG', 2: 'I-WRONG'}


In [ ]:
# Configurações de Treinamento

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",       # Estratégia de save igual à de eval
    load_best_model_at_end=True, # Salvamento e carregamento do melhor modelo ao final
    metric_for_best_model="f0_5",  # O critério é o F0.5 Score
    fp16=True,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=3,         # 3 épocas
    weight_decay=0.01,
    logging_dir="./logs",
    report_to=[],
)

In [ ]:
# Cria o data collator que faz padding dinâmico
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
)

In [ ]:
import os
import gc
import shutil
import torch
import pandas as pd
from datasets import disable_caching
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments

# --- OTIMIZAÇÃO 1: Desativa completamente o cache em disco do HF Datasets ---
disable_caching()

nome_modelo = "neuralmind/bert-base-portuguese-cased"
num_experimentos = 30
resultados_experimentos = []

# --- OTIMIZAÇÃO 2: Definir diretório temporário LOCAL (fora do Google Drive) ---
diretorio_local_temp = "/content/temp_results"

print(f"*** INÍCIO DO CICLO DE {num_experimentos} EXPERIMENTOS COM BERT BASE ***")

for i in range(num_experimentos):
    current_seed = 100 + i

    # --- OTIMIZAÇÃO 3: Split feito 100% em RAM ---
    ds_split = full_dataset.shuffle(seed=current_seed).train_test_split(
        test_size=0.2,
        seed=current_seed,
        keep_in_memory=True  # Impede a criação de arquivos temporários de split no disco
    )

    # --- OTIMIZAÇÃO 4: Tokenização dinâmica mantida em RAM ---
    tokenized_ds = ds_split.map(
        lambda batch: tokenize_and_align_labels(batch, tokenizer, label_all_subtokens=False),
        batched=True,
        load_from_cache_file=False,
        keep_in_memory=True  # Impede a criação de arquivos de cache Arrow gigantescos no disco
    )

    # Reinicialização do Modelo (Essencial para independência estatística)
    model = AutoModelForTokenClassification.from_pretrained(
        nome_modelo,
        num_labels=len(lista_rotulos),
        id2label=id_to_label,
        label2id=label_to_id
    ).to(device)

    # O diretório de saída agora aponta para o disco temporário local
    output_dir_atual = os.path.join(diretorio_local_temp, f"exp_{i+1}")

    training_args = TrainingArguments(
        output_dir=output_dir_atual,
        eval_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=4,      # Batch size reduzido para evitar Out Of Memory
        gradient_accumulation_steps=4,      # Acumula gradientes para simular batch de 16
        num_train_epochs=3,
        weight_decay=0.01,
        disable_tqdm=True,
        report_to=[],
        fp16=torch.cuda.is_available(),
        save_strategy="no",                 # Não salva checkpoints de pesos no disco
        logging_strategy="no",              # OTIMIZAÇÃO 5: Não escreve arquivos de log no disco
        save_total_limit=1
    )

    # Trainer configurado dentro do loop
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # Execução do treino
    trainer.train()

    # Coleta de métricas essenciais
    metrics = trainer.evaluate(eval_dataset=tokenized_ds["test"])
    resultados_experimentos.append({
        'Precision': metrics['eval_precision'],
        'Recall': metrics['eval_recall'],
        'F0.5-Score': metrics['eval_f0_5'],
        'Accuracy': metrics['eval_accuracy']
    })

    print(f"✅ Rodada {i+1}/30 finalizada (Seed: {current_seed}) | F0_5: {metrics['eval_f0_5']:.4f}")

    # --- OTIMIZAÇÃO 6: Limpeza profunda de memória RAM e VRAM a cada rodada ---
    del model
    del trainer
    del ds_split
    del tokenized_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Remove a pasta de logs local imediatamente
    if os.path.exists(output_dir_atual):
        shutil.rmtree(output_dir_atual)

# Remove a pasta mãe temporária local completamente do disco local
if os.path.exists(diretorio_local_temp):
    shutil.rmtree(diretorio_local_temp)


# --- CÁLCULO E GERAÇÃO DO ARQUIVO FINAL ---
df_resultados = pd.DataFrame(resultados_experimentos)

# 1. Calcula as estatísticas
stats_media = df_resultados.mean()
stats_desvio = df_resultados.std()

# 2. Cria o DataFrame de estatísticas
df_stats = pd.DataFrame([stats_media, stats_desvio], index=['Média', 'Desvio Padrão'])

# 3. Une tudo em um DataFrame final
df_final = pd.concat([df_resultados, df_stats])

# --- OTIMIZAÇÃO 7: Salvar o CSV final no Google Drive ---
caminho_salvamento_drive = "/content/drive/MyDrive/Colab Notebooks/resultados_bertbase.csv"
df_final.to_csv(caminho_salvamento_drive, index=True)

print(f"\nOs resultados dos 30 experimentos foram salvos com sucesso em: '{caminho_salvamento_drive}'")


# --- Relatório Estatístico na Tela ---
print("\n" + "="*45)
print("📊 RELATÓRIO ESTATÍSTICO DE SPANS (n=30)")
print("="*45)
for metrica in df_resultados.columns:
    print(f"{metrica:10}: {stats_media[metrica]:.4f} (± {stats_desvio[metrica]:.4f})")
print("="*45)

*** INÍCIO DO CICLO DE 30 EXPERIMENTOS COM BERT BASE ***


Map:   0%|          | 0/15400 [00:00<?, ? examples/s]

Map:   0%|          | 0/3850 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

{'train_runtime': '556.4', 'train_samples_per_second': '83.04', 'train_steps_per_second': '5.193', 'train_loss': '0.9686', 'epoch': '3'}
{'eval_loss': '0.3039', 'eval_precision': '0.3777', 'eval_recall': '0.2054', 'eval_f0_5': '0.3234', 'eval_accuracy': '0.9192', 'eval_runtime': '11.13', 'eval_samples_per_second': '346', 'eval_steps_per_second': '43.32', 'epoch': '3'}
✅ Rodada 29/30 finalizada (Seed: 128) | F0_5: 0.3234


Map:   0%|          | 0/15400 [00:00<?, ? examples/s]

Map:   0%|          | 0/3850 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'train_runtime': '556.8', 'train_samples_per_second': '82.98', 'train_steps_per_second': '5.189', 'train_loss': '0.9787', 'epoch': '3'}
{'eval_loss': '0.3092', 'eval_precision': '0.3953', 'eval_recall': '0.2284', 'eval_f0_5': '0.3449', 'eval_accuracy': '0.9198', 'eval_runtime': '10.89', 'eval_samples_per_second': '353.7', 'eval_steps_per_second': '44.28', 'epoch': '3'}
✅ Rodada 30/30 finalizada (Seed: 129) | F0_5: 0.3449

Os resultados dos 30 experimentos foram salvos com sucesso em: '/content/drive/MyDrive/Colab Notebooks/resultados_bertbase.csv'

📊 RELATÓRIO ESTATÍSTICO DE SPANS (n=30)
Precision : 0.3894 (± 0.0132)
Recall    : 0.2218 (± 0.0082)
F0.5-Score: 0.3382 (± 0.0103)
Accuracy  : 0.9219 (± 0.0023)


In [ ]:
# Cálculo das métricas na memória
stats_media = df_resultados.mean()
stats_desvio = df_resultados.std()

# Opcional: Salvar a tabela consolidada direto em CSV para não perder
import pandas as pd
df_estatisticas = pd.DataFrame({
    'Média': stats_media,
    'Desvio_Padrão': stats_desvio
})
df_estatisticas.to_csv("estatisticas_bertbase.csv", index=True)
print("Estatísticas calculadas e salvas com sucesso!")

Estatísticas calculadas e salvas com sucesso!
